#

# Introduction au Nutriscope


In [1]:
# Installation (si nécessaire)
#%pip install pandas
#%pip install --upgrade IPython
#%pip install pyarrow

# Import de la bibliothèque
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import time

## Basic Infos

In [2]:
lang = None
if lang == None:
    lang = input("Which language would you like to use?")
    lang = lang.lower()
    lang = lang[:2]

In [4]:
#chargement
#Le dump complet fait plusieurs gigaoctets : filtrer à la source (colonnes, pays) fait partie de l'exercice. 
# Prévoir du réseau et de la patience — c'est la première vraie contrainte du projet.
# https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html

# Create a Path object
file_path = Path(f"../data/{lang.upper()}/food-{lang}.parquet.gzip")
food_df = None
start = time.process_time()

# Check if the file exists
if file_path.exists():
    print("-------------- Import Existing --------------")
    food_df = pd.read_parquet(file_path, engine="pyarrow") #1min9.9s
    print("read_parquet : ", time.process_time() - start)
    start = time.process_time()
    food_df = pq.read_table(file_path, use_threads=False)
    print("read_table (thread == False): ", time.process_time() - start)
    start = time.process_time()
    food_df = pq.read_table(file_path, use_threads=True)
    print("read_table (thread == True): ", time.process_time() - start)
else:
    try:
        print("-------------- Import New --------------")
        food_df = pd.read_parquet("../data/food.parquet", engine="pyarrow", filters=[("lang", "==", lang)])#5min15s / 319ms
        print(time.process_time() - start)
        start = time.process_time()
        print("-------------- Export New --------------")
        os.mkdir(f"../data/{lang.upper()}/")
        food_df.to_parquet(file_path, compression='gzip')# 712ms
        print(time.process_time() - start)
    except Exception as e:
        print(e)

#dimensions
print(f"Dimensions : {food_df.shape}")
#types
print(f"Types : {food_df.dtypes}")
#mémoire
print(f"Mémoire : {food_df.info}")
display(food_df)

-------------- Import Existing --------------
read_parquet :  91.1875
read_table (thread == False):  44.359375
read_table (thread == True):  41.25
Dimensions : (1338651, 111)


AttributeError: 'pyarrow.lib.Table' object has no attribute 'dtypes'

In [11]:
#food_df.columns.tolist()
food_df["countries_tags"]


0           [en:france]
1           [en:france]
2           [en:france]
3           [en:france]
4           [en:france]
               ...     
1338646     [en:france]
1338647     [en:france]
1338648    [en:ireland]
1338649    [en:tunisia]
1338650     [en:france]
Name: countries_tags, Length: 1338651, dtype: object

In [ ]:
#taux de remplissage par colonne

## Explore

In [ ]:
#1 combien de produits vendus en France ? -Sacha


In [ ]:
#2 quelle  part a un Nutri-Score renseigné? -Clement
food_df.groupby("nutriscore_grade").size()

mode_paiment = df["mode_paiement"].value_counts()
axes[1,0].pie(mode_paiment, labels=mode_paiment.index, autopct='%1.1f%%', startangle=90)
axes[1,0].set_title('Modes de paiements', fontweight='bold')

nutriscore_grade
a                  66254
b                  53087
c                 102494
d                 128965
e                 135625
not-applicable     43103
unknown           805396
dtype: int64

In [ ]:
#3 les dix marques les plus présentes ? -Sacha

In [12]:
#4 le taux de manquants sur les nutriments clés ( energy_100g , sugars_100g , salt_100g ) ? -Clement
#print(food_df.isna())
percent_missing = food_df.isnull().sum() * 100 / len(food_df)
missing_value_df = pd.DataFrame({'column_name': food_df.columns,
                                 'percent_missing': percent_missing})

missing_value_df.groupby("nutriment").agg({
    "salaire": ["energy_100g", "sugars_100g", "salt_100g"]
}).round(0)
display(missing_value_df)

mode_paiment = df["mode_paiement"].value_counts()
axes[1,0].pie(mode_paiment, labels=mode_paiment.index, autopct='%1.1f%%', startangle=90)
axes[1,0].set_title('Modes de paiements', fontweight='bold')

KeyError: 'nutriment'

#5 qu'est-ce qui vous semble le plus « sale » dans ces données ?

+ La façon de gérer les languges et les pays est chaotique. Il n'y a pas de distinction entre le français de France et Belgique par exemple ce qui oblige un travail supplémentaire.
+ Au regard du retour de la commande sur le nutri-score, on remarque que plusieurs collaborateurs ont ajouté des données sans respecté un standard (not-applicable, unknown, null, None).